In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
from IPython.display import display, HTML

# Define runs to analyze
run_names = ['run_017', 'run_018', 'run_019']
#run_names = ['run_010', 'run_015']
runs_dir = '../runs'

all_data = []
for run in run_names:
    run_path = os.path.join(runs_dir, run)
    if not os.path.exists(run_path):
        print(f'Run directory {run_path} does not exist')
        continue
        
    # Load all event files within the run directory (including subdirectories for eval)
    event_files = sorted(glob.glob(os.path.join(run_path, '**', 'events.out.tfevents.*'), recursive=True))
    print(f'Found {len(event_files)} event files for {run}')
    
    for ev_file in event_files:
        # Size guidance to ensure we load enough scalars. 0 means load all.
        size_guidance = {'scalars': 0}
        ea = event_accumulator.EventAccumulator(ev_file, size_guidance=size_guidance)
        ea.Reload()
        
        if 'scalars' in ea.Tags():
            tags = ea.Tags()['scalars']
            for tag in tags:
                for ev in ea.Scalars(tag):
                    all_data.append({'run': run, 'tag': tag, 'step': ev.step, 'value': ev.value})

if all_data:
    df = pd.DataFrame(all_data)
    # Remove duplicates by taking the last value for a given (run, tag, step)
    df = df.groupby(['run', 'tag', 'step']).last().reset_index()
    print('\nData successfully loaded and aggregated.')
else:
    print('\nNo data found in the specified runs.')
    df = pd.DataFrame(columns=['run', 'tag', 'step', 'value'])

Found 3 event files for run_017
Found 2 event files for run_018
Found 2 event files for run_019

Data successfully loaded and aggregated.


In [2]:
# Extract Custom evaluation metrics
eval_tags = [t for t in df['tag'].unique() if 'mAP' in t or 'eval' in t]
eval_tags = [t for t in eval_tags if t not in ['eval/loss']]

if not eval_tags:
    print("No evaluation metrics (mAP) found in the logs.")
else:
    # Compute best metrics per run
    best_metrics = []
    for tag in eval_tags:
        for run in run_names:
            run_data = df[(df['tag'] == tag) & (df['run'] == run)]
            if not run_data.empty:
                max_val = run_data['value'].max()
                best_metrics.append({'Metric': tag, 'Run': run, 'Best Value': max_val})
                
    best_df = pd.DataFrame(best_metrics)

    # Load hierarchy
    import json
    import plotly.express as px

    hierarchy_path = '../hierarchy.json'
    hierarchy = {}
    if os.path.exists(hierarchy_path):
        with open(hierarchy_path, 'r') as f:
            hierarchy = json.load(f)

    groups = {'Global Metrics': [t for t in eval_tags if t.startswith('mAP/') or not t.startswith('mAP_Class/')]} 
    
    for group_name, classes in hierarchy.items():
        # The tags in the logs are like 'mAP_Class/note'
        group_tags = [f'mAP_Class/{cls}' for cls in classes if f'mAP_Class/{cls}' in eval_tags]
        if group_tags:
            groups[group_name] = group_tags
            
    # Ensure any ungrouped mAP_Class tags are not lost
    grouped_tags = set(sum(groups.values(), []))
    ungrouped = [t for t in eval_tags if t not in grouped_tags]
    if ungrouped:
        groups['Other Classes'] = ungrouped

    # Generate tables and plots per group
    for group_name, tags in groups.items():
        group_df = best_df[best_df['Metric'].isin(tags)].copy()
        if group_df.empty:
            continue
            
        # 1. Summary Table
        summary_df = group_df.pivot(index='Metric', columns='Run', values='Best Value')
        for r in run_names:
            if r not in summary_df.columns:
                summary_df[r] = np.nan
                
        def highlight_max(s):
            is_max = s == s.max()
            return ['font-weight: bold; background-color: #d4edda; color: black' if v else '' for v in is_max]
            
        styled_df = (
            summary_df.style
            .apply(highlight_max, axis=1)
            .format("{:.4f}", na_rep="-")
            .set_caption(f"{group_name.replace('_', ' ')} - Maximum Performance by Model")
            .set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'left'), ('font-size', '14px')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('font-size', '13px')]},
                {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
            ])
        )
        display(HTML(styled_df.to_html()))
        
        # 2. Plotly Plot
        # Strip the prefix for cleaner legends and ticks
        group_df['Metric Name'] = group_df['Metric'].str.replace('mAP_Class/', '').str.replace('mAP/', '')
        
        fig = px.line(group_df, x='Run', y='Best Value', color='Metric Name', symbol='Metric Name',
                      markers=True, title=f'{group_name.replace("_", " ")} - Plot Comparison',
                      template='plotly_white')
        # Add some padding to Y axis so markers are not cut off
        fig.update_yaxes(range=[max(0, group_df['Best Value'].min()-0.05), min(1.05, group_df['Best Value'].max()+0.05)])
        fig.update_traces(marker=dict(size=12), line=dict(width=2))
        fig.update_layout(xaxis_title='Run', yaxis_title='Best mAP Score', hovermode='x unified')
        fig.show()


Run,run_017,run_018,run_019
Metric,,,
mAP/IoU_0.50,0.5957,0.6426,0.6469
mAP/IoU_0.50_0.95,0.4684,0.5302,0.5288
mAP/IoU_0.75,0.5193,0.5823,0.5781


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/chord,0.4761,0.5166,0.5276
mAP_Class/mRest,0.3954,0.2141,0.1999
mAP_Class/note,0.4835,0.4927,0.4965
mAP_Class/notehead,0.5629,0.5631,0.5641
mAP_Class/rest,0.7543,0.7487,0.7742


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/barLine,0.3872,0.4742,0.5616
mAP_Class/clef,0.9688,0.9824,0.9839
mAP_Class/keyAccid,0.9834,0.9678,0.9691
mAP_Class/keySig,0.6392,0.6281,0.6285
mAP_Class/ledgerLines,0.0211,0.0418,0.0492
mAP_Class/measure,0.5008,0.5088,0.5018
mAP_Class/meterSig,0.8760,0.9373,0.9566
mAP_Class/staff,0.7336,0.7237,0.6982


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/beam,0.6634,0.6969,0.7093
mAP_Class/flag,0.6299,0.6031,0.5966
mAP_Class/slur,0.3943,0.4623,0.5061
mAP_Class/stem,0.5810,0.5999,0.6077
mAP_Class/tie,0.5244,0.5464,0.6100
mAP_Class/tuplet,0.0843,0.1082,0.1451
mAP_Class/tupletNum,0.7491,0.8862,0.8988


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/arpeg,0.0000,0.0019,0.0005
mAP_Class/artic,0.5074,0.5705,0.6530
mAP_Class/fermata,0.6277,0.4320,0.6338
mAP_Class/mordent,0.0000,0.1592,0.2478
mAP_Class/trill,0.0000,0.5589,0.1476


...

Run,run_017,run_018,run_019
Metric,,,
mAP_Class/label,0.1991,0.2279,0.2268
mAP_Class/labelAbbr,0.7570,0.7928,0.7918
mAP_Class/mNum,-,-1.0000,-1.0000
mAP_Class/pgHead,0.9006,0.8766,0.8986


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/grpSym,0.8794,0.9238,0.9257
mAP_Class/repeatMark,0.3127,0.4092,0.4517
mAP_Class/voltaBracket,0.3140,0.5306,0.5995


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/fing,0.0000,0.0343,0.0540
mAP_Class/syl,0.3231,0.3868,0.3977
mAP_Class/verse,0.4642,0.4671,0.4765


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/autogenerated,0.8481,0.6221,0.8686
mAP_Class/layer,0.3745,0.4117,0.4175
mAP_Class/page-margin,0.6953,0.8214,0.8338
mAP_Class/system,0.7612,0.7860,0.7297


Run,run_017,run_018,run_019
Metric,,,
mAP_Class/pedal,0.4849,0.5518,0.5960
mAP_Class/turn,0.0000,0.2398,0.2147


In [3]:
best_df.pivot(index='Metric', columns='Run', values='Best Value').to_markdown()

'| Metric                  |     run_017 |      run_018 |      run_019 |\n|:------------------------|------------:|-------------:|-------------:|\n| mAP/IoU_0.50            |   0.595708  |  0.642563    |  0.646895    |\n| mAP/IoU_0.50_0.95       |   0.468359  |  0.530163    |  0.528796    |\n| mAP/IoU_0.75            |   0.519337  |  0.582332    |  0.578068    |\n| mAP_Class/accid         |   0.835931  |  0.857493    |  0.877588    |\n| mAP_Class/arpeg         |   0         |  0.00190275  |  0.000503564 |\n| mAP_Class/artic         |   0.507394  |  0.570547    |  0.652972    |\n| mAP_Class/autogenerated |   0.848145  |  0.622132    |  0.868621    |\n| mAP_Class/barLine       |   0.387212  |  0.474193    |  0.561628    |\n| mAP_Class/beam          |   0.66337   |  0.696914    |  0.709325    |\n| mAP_Class/chord         |   0.476084  |  0.516634    |  0.527589    |\n| mAP_Class/clef          |   0.968824  |  0.982447    |  0.983857    |\n| mAP_Class/dir           |   0.0432675 |  0.13275